# 📈 S&P 500 Pairs Trading 交易期 (Trading Period) 策略邏輯與公式詳解

## 📝 概述
在配對交易中，**交易期 (Trading Period)**（預設 $T = 126$ 天）的核心任務是**根據即時信號執行交易，並結合高階部位與全域風控機制管理多週期 Slots 權益**。

> [!IMPORTANT]
> **本文件完全以 `strategies/trading/` 下實際運行的 `.py` 原始碼為準**進行解析，糾正了舊版 Notebook 文檔中的過時描述。

### 📂 策略檔案結構對照：
- **Z-Score 狀態機交易核心** $\rightarrow$ `strategies/trading/zscore_trading.py` (包含 SSD 與 HDBSCAN 的交易實作)
- **純 DTW 交叉帶內進場交易** $\rightarrow$ `strategies/trading/pure_dtw_trading.py`
- **深度強化學習 LSTM 交易** $\rightarrow$ `strategies/trading/drl_lstm_trading.py`

---

## 🧠 一、 通用 Z-Score 狀態機交易期邏輯 (`zscore_trading.py`)

大部分 Z-Score 基礎策略均繼承或使用 `zscore_trading.py` 中實現的狀態機。

### 1.1 價差重構與 Z-Score 計算
- **路徑 A：OLS 對數空間 (如 HDBSCAN / OLS SSD 策略)**：
  當 `ols_alpha is not None`，使用對數價格 $\ln(P)$ 及形成期傳入的常數項 $\alpha$、避險比例 $\beta$：
  $$\text{Spread}_t = \ln(P_{A, t}) - \alpha - \beta \cdot \ln(P_{B, t})$$
- **路徑 B：標準化價格空間 (如 Basic SSD 策略)**：
  當 `ols_alpha is None`，在標準化價格/對數價格空間下計算價差：
  $$\text{Spread}_t = P'_{A, t} - \beta \cdot P'_{B, t}$$
- **Z-Score 計算 (固定 vs 滾動)**：
  - 固定模式 (`zscore_window = 0`)：$Z_t = \frac{\text{Spread}_t - \mu_{form}}{\sigma_{form}}$。
  - 滾動模式 (`zscore_window = W`)：在交易期滾動視窗 $W$ 天內重新進行 OLS 回歸得到動態 $\alpha_t, \beta_t$，並以殘差變異數作為標準差中心：
    $$\sigma_{residual, t} = \sqrt{\max\left(\text{Var}_W(P'_A) - \beta_t \cdot \text{Cov}_W(P'_A, P'_B), 0\right)}$$
    $$Z_t = \frac{\text{Spread}_t}{\sigma_{residual, t}}$$

### 1.2 進出場訊號
- **建倉 (Entry)**：當 $\vert Z_t \vert > \text{entry\_z}$ 時。
  - $Z_t > \text{entry\_z} \rightarrow$ 空頭建倉 (空 A 多 B)。
  - $Z_t < -\text{entry\_z} \rightarrow$ 多頭建倉 (多 A 空 B)。
- **平倉 (Exit)**：當 $Z_t$ 回歸至零軸邊界時，即 $\vert Z_t \vert \le \text{exit\_z}$。

### 1.3 資金部位風險中性配置
股票 A 與 B 的市值部位 $v_a, v_b$ 根據避險比例 $\beta$ 進行加權分配（對沖市場 Beta 風險）：
$$\text{Total Weight} = 1.0 + |\beta|$$
$$v_a = C_{pair} \times \frac{1.0}{\text{Total Weight}}, \quad v_b = C_{pair} \times \frac{|\beta|}{\text{Total Weight}}$$
*(註：在 `ssd_basic.py` 中因 $\beta = 1.0$，部位資金分配為等額的 50%/50% 分配)*

## ⏳ 二、 純 DTW 交叉帶內進場交易邏輯 (`pure_dtw_trading.py`)

純 DTW 交易策略繼承自 `Trading` (來自 `zscore_trading.py`)，但在**開倉訊號上做出了重要調整，以減少逆勢建倉風險**。

### 2.1 交叉帶內進場條件 (Cross Back Inside the Bands)
傳統策略在 Z-Score「突破」臨界線時立刻進場，容易遇到「價差持續發散」的單邊風險。純 DTW 策略要求**價差 Z-Score 必須先突破，並在「回折交叉回歸帶內」時才觸發建倉**：
- **空頭建倉 (Short Entry, -1)**：當前一日 $Z_{t-1} > \text{entry\_z}$ 且當日 $Z_t \le \text{entry\_z}$ 時。（從上方折返穿過上開倉線）
- **多頭建倉 (Long Entry, 1)**：當前一日 $Z_{t-1} < -\text{entry\_z}$ 且當日 $Z_t \ge -\text{entry\_z}$ 時。（從下方折返穿過下開倉線）

### 2.2 均值平倉條件
- **空頭平倉 (Short Exit)**：當 $Z_t \le \text{exit\_z}$ 時。
- **多頭平倉 (Long Exit)**：當 $Z_t \ge -\text{exit\_z}$ 時。
- **資金配置**：由於純 DTW 策略假設兩隻正規化收益率指數等價，其對沖比例固定為 $1.0$，資金分配為 50%/50% 等權重配置。

## 🤖 三、 深度強化學習 LSTM 交易期邏輯 (`drl_lstm_trading.py`)

此策略將配對交易建模為馬可夫決策過程 (MDP)，利用 **DQN + LSTM** 網路，在複雜的多維特徵空間下學習最佳建平倉策略。

### 3.1 Gymnasium 環境特徵空間 (5維狀態)
在交易期每日，環境提取以下 5 維狀態向量 $S_t$ 作為 Agent 的觀察值：
1. **Spread Z-Score**：即時價差標準化值。
2. **相對回報率 (Rel_Return)**：兩隻股票日收益率之差：$R_{A,t} - R_{B,t}$。
3. **均線距離 (MA_Dist)**：即時價差相對於其 20 日移動平均線的乖離度：$\text{Spread}_t - \text{MA}_{20}(\text{Spread}_t)$。
4. **剩餘交易時間比率 (Time-to-Maturity)**：離交易期結束的剩餘時間比例：$\frac{T - t}{T}$，用以提示臨近強制平倉的風險。
5. **動態波動率 (Volatility)**：價差的 20 日滾動標準差，捕捉即時市場風險層級。

### 3.2 動作空間與獎勵函數 (Reward Function)
- **離散動作空間**：`0` (Flat/平倉或空倉), `1` (Long Spread), `2` (Short Spread)。
- **事件型獎勵 (Reward)**：
  - **平倉回報**：平倉時根據該筆交易的累計淨收益率給予對應的正/負獎勵：
    $$\text{Reward}_{exit} = \frac{\text{Trade PnL}}{\text{Capital}} \times 100.0$$
  - **交易摩擦懲罰 (Action Penalty)**：開倉時，扣除開倉摩擦成本比例作為懲罰，以抑制無效過度交易：
    $$\text{Reward}_{entry} = -\frac{\text{Entry Fee}}{\text{Capital}} \times 100.0 \times 0.5$$

### 3.3 LSTM_DQN 模型與時序記憶
- **時序序列輸入**：Agent 的輸入並非單日的 5 維特徵，而是**過去 $seq\_len = 10$ 天的時序特徵序列矩陣** (形狀為 $10 \times 5$)。
- **神經網路**：通過 PyTorch 的 `nn.LSTM` 提取這 10 天特徵的動態時序記憶，取最後一個隱藏狀態 (Last Hidden State) 輸入全連接層，輸出 3 個動作對應的 Q 值。

### 3.4 形成期預訓練 + 交易期實時推論
- 在每個交易期開始之前，Agent 會先在其**形成期的 252 天歷史數據環境上進行 100 輪 (Episodes) 的 DQN 強化學習訓練**，學習該配對專屬的套利規律。
- 進入交易期後，將探索率 $\epsilon$ 降為 0，模型進行實時 DQN 推論並執行交易。

In [ ]:
# drl_lstm_trading.py 中 Gymnasium 環境與 PyTorch LSTM_DQN 網路實作
import torch
import torch.nn as nn
import numpy as np

class LSTM_DQN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, output_dim=3, num_layers=1):
        super(LSTM_DQN, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        # batch_first=True, 輸入形狀為 (batch_size, seq_len, input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # out 形狀: (batch_size, seq_len, hidden_dim)
        out, _ = self.lstm(x)
        # 取最後一個時間步 (last time-step) 的隱藏狀態
        out = out[:, -1, :]
        # 輸出 3 個動作的 Q 值
        return self.fc(out)

def get_trading_env_observation(zscore_series, price_a, price_b, current_step, max_steps):
    # 模擬 drl_lstm_trading.py 中的 5 維環境觀察值生成
    zscore = zscore_series[current_step]
    
    # 相對報酬率
    ret_a = (price_a[current_step] - price_a[current_step-1]) / price_a[current_step-1] if current_step > 0 else 0.0
    ret_b = (price_b[current_step] - price_b[current_step-1]) / price_b[current_step-1] if current_step > 0 else 0.0
    rel_return = ret_a - ret_b
    
    # 均線乖離率 (相對 20日均線)
    ma20 = np.mean(zscore_series[max(0, current_step-20):current_step+1])
    ma_dist = zscore - ma20
    
    # 剩餘到期時間比率
    time_to_maturity = (max_steps - current_step) / max_steps
    
    # 波動度
    volatility = np.std(zscore_series[max(0, current_step-20):current_step+1]) if current_step > 1 else 1.0
    
    return np.array([zscore, rel_return, ma_dist, time_to_maturity, volatility], dtype=np.float32)

## 🧠 四、 共享部位管理與六大風控機制

本量化系統實作了完善的六大風控防線，防範黑天鵝事件與逆勢單邊發散風險：
1. **個股單筆停損 (`SL`)**：當個股未實現虧損比例達到 `stop_loss_pct` 時，觸發強制平倉並凍結該配對。
   $$\text{Loss Ratio} = -\frac{\text{Trade PnL}}{C_{pair}} \ge \text{stop\_loss\_pct}$$
2. **部位動態 Z-Score 偏離停損 (`DSZ`)**：當 $\vert Z_t \vert > \text{dynamic\_stop\_z}$（如 3.0 或 5.0）時，判斷發生結構性破裂 (Structural Break) 即刻停損。
3. **全域投資組合層級最大回撤停損 (`PSL`)**：當總資金虧損達到 `portfolio_stop_loss_pct` (如 10%) 時，觸發 PSL 一鍵斬倉所有持倉配對並重置且凍結所有交易。
4. **產業分散化集中度上限 (`MSR`)**：限制單一產業的配對數量上限：$\max(1, \lfloor N \cdot \text{max\_sector\_ratio} \rfloor)$。
5. **方向性建倉冷卻機制 (Cooldown Period)**：平倉後進入冷卻，多頭平倉需等 $Z_t \ge -\text{EXIT\_Z}$，空頭平倉需等 $Z_t \le \text{EXIT\_Z}$ 才能解凍。
6. **自適應波動率調節機制 (`VOL ADJ`)**：依近期 20 日波動率放大形成期標準差，防止無序震盪中頻繁建倉：
   $$\sigma_{adjusted} = \sigma_{formation} \times \max\left(1.0, \frac{\sigma_{roll20}}{\sigma_{formation}}\right)$$

## 📊 五、 所有策略交易期特徵與參數對比總結

| 交易特徵 | 經典 Z-Score 交易 | 純 DTW 交叉交易 | DRL LSTM 強化學習交易 |
| :--- | :--- | :--- | :--- |
| **對應檔案** | `zscore_trading.py` | `pure_dtw_trading.py` | `drl_lstm_trading.py` |
| **開倉條件** | Z-Score 絕對值突破開倉線 ($\vert Z_t \vert > \text{entry\_z}$) | Z-Score 從帶外折返交叉回歸帶內 | DRL 代理人實時輸出動作 `1` 或 `2` |
| **平倉條件** | Z-Score 回歸到退出區間 ($\vert Z_t \vert \le \text{exit\_z}$) | Z-Score 回歸至均值中心 | DRL 代理人實時輸出動作 `0` (Flat) |
| **部位與對沖** | 依 OLS $\beta$ 進行風險中性加權配置 | 固定為 1.0 (等權重 50%/50% 分配) | 依 OLS $\beta$ 進行風險中性加權配置 |
| **特徵狀態** | 僅依據單一的 Spread Z-Score 數值 | 僅依據單一的 Spread Z-Score 數值 | 5維狀態空間（Z-Score, 相對回報, 均線距離, 到期剩餘時間, 波動率） |
| **時序記憶** | 無 | 前一日與當日 Z-Score 比較 ($Z_{t-1}, Z_t$) | 過去 10 天時序特徵矩陣，經由 LSTM 網路提煉 |
| **預訓練機制** | 無 | 無 | 在形成期歷史數據上預訓練 100 輪 (Episodes) |
| **部位風控** | 完整支援 (SL, DSZ, PSL, MSR, VOL ADJ) | 完整支援 (SL, DSZ, PSL, MSR, VOL ADJ) | 支援環境內摩擦成本懲罰與強平機制 |

---